In [ ]:
import math
import statistics
import requests


OPEN_ELEVATION_URL = "https://api.open-elevation.com/api/v1/lookup"


def meters_to_deg_lat(meters: float) -> float:
    # Approx: 1 deg latitude ~= 111,320 meters
    return meters / 111_320.0


def meters_to_deg_lon(meters: float, lat_deg: float) -> float:
    # Approx: 1 deg longitude ~= 111,320 * cos(latitude) meters
    return meters / (111_320.0 * math.cos(math.radians(lat_deg)))


def make_grid_points(lat: float, lon: float, side_m: float = 100.0, grid_n: int = 5):
    """
    Build an NxN grid of points covering a side_m x side_m square centered at (lat, lon).
    grid_n=5 => 25 points. Includes edges.
    """
    half = side_m / 2.0
    if grid_n < 2:
        raise ValueError("grid_n must be >= 2 so the square has area coverage.")

    # Evenly spaced offsets from -half to +half (meters)
    step = side_m / (grid_n - 1)
    offsets = [(-half + i * step) for i in range(grid_n)]

    locations = []
    for dy in offsets:  # north/south (lat)
        dlat = meters_to_deg_lat(dy)
        for dx in offsets:  # east/west (lon)
            dlon = meters_to_deg_lon(dx, lat)
            locations.append({"latitude": lat + dlat, "longitude": lon + dlon})

    return locations

In [ ]:
import requests

URL = "https://api.opentopodata.org/v1/mapzen"

locations = [
    {"latitude": -35.2809, "longitude": 149.1300},
    {"latitude": -35.2812, "longitude": 149.1305},
    {"latitude": -35.2815, "longitude": 149.1310},
]

resp = requests.post(URL, json={"locations": locations}, timeout=20)
resp.raise_for_status()

data = resp.json()
print(data)  # {"results":[{"latitude":...,"longitude":...,"elevation":...}, ...]}

elevations = [r["elevation"] for r in data["results"]]
print("elevations:", elevations)
print("avg:", sum(elevations) / len(elevations))

In [ ]:
import requests

def get_elevation(lat, lon):
    """
    Fetch elevation for a given latitude and longitude using Open-Elevation API.
    
    Args:
        lat (float): Latitude in decimal degrees.
        lon (float): Longitude in decimal degrees.
    
    Returns:
        float: Elevation in meters, or None if request fails.
    """
    # Validate input
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ValueError("Invalid latitude or longitude values.")

    url = "https://api.opentopodata.org/v1/mapzen"
    params = {"locations": f"{lat},{lon}"}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()  # Raise HTTPError for bad responses
        data = response.json()

        # Extract elevation
        if "results" in data and len(data["results"]) > 0:
            return data["results"][0]["elevation"]
        else:
            print("No elevation data found.")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error fetching elevation: {e}")
        return None



# Example coordinates: Mount Everest summit
latitude = 27.9881
longitude = 86.9250

elevation = get_elevation(latitude, longitude)
if elevation is not None:
    print(f"Elevation at ({latitude}, {longitude}): {elevation} meters")
else:
    print("Could not retrieve elevation.")


In [ ]:
requests.get("https://api.opentopodata.org/v1/mapzen?locations=57.688709,11.976404", timeout=10)

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
import requests

# Call API outside Snowflake
resp = requests.get("https://api.opentopodata.org/v1/mapzen?locations=57.688709,11.976404")
data = resp.json()

# Write to Snowflake
df = session.create_dataframe(data)
df.write.save_as_table("api_results")


In [ ]:
!pip install fastkml geopandas shapely

In [ ]:
import zipfile
from fastkml import kml
# Path to the KMZ file
kmz_file_path = "data/A_reg_WMS_nobor_m.kmz"
# Extract the KML content from the KMZ file
with zipfile.ZipFile(kmz_file_path, 'r') as kmz:
   kml_content = kmz.read('doc.kml') # 'doc.kml' is the default file inside KMZ
# Parse the KML content using fastkml
k = kml.KML()
k.from_string(kml_content)
# Access features (e.g., placemarks, polygons)
features = list(k.features())
for feature in features:
   print(feature.name) # Example: Print feature names

In [ ]:
import zipfile
from pathlib import Path
from fastkml import kml
import geopandas as gpd
from shapely.geometry import shape

def process_kmz(kmz_path):
    kmz_path = Path(kmz_path)
    if not kmz_path.exists() or kmz_path.suffix.lower() != ".kmz":
        raise FileNotFoundError(f"Invalid KMZ file: {kmz_path}")

    # Step 1: Extract KML from KMZ
    with zipfile.ZipFile(kmz_path, 'r') as kmz:
        kml_files = [f for f in kmz.namelist() if f.endswith('.kml')]
        print(kml_files)
        if not kml_files:
            raise ValueError("No KML file found inside KMZ.")
        kml_content = kmz.read(kml_files[0]).decode('utf-8')

    # Step 2: Parse KML
    k = kml.KML.parse(kml_content)
    # k.from_string(kml_content)
    # print(k.to_string(prettyprint=True, precision=6))

    # Step 3: Extract features into GeoDataFrame
    features = []
    for document in k.features():
        for folder in document.features():
            for placemark in folder.features():
                geom = placemark.geometry
                print(geom)
                if geom:
                    features.append({
                        "name": placemark.name,
                        "geometry": shape(geom)
                    })

    if not features:
        raise ValueError("No geometries found in KML.")

    gdf = gpd.GeoDataFrame(features, crs="EPSG:4326")
    return gdf



gdf = process_kmz("data/A_reg_WMS_nobor_m.kmz")
print(gdf.head())  # Preview data
gdf.to_file("output.geojson", driver="GeoJSON")  # Save as GeoJSON
print("GeoJSON saved as output.geojson")


In [ ]:

def get_children(obj):
    """Return children features regardless of whether .features is list or method."""
    if not hasattr(obj, "features"):
        return []
    f = obj.features
    return list(f()) if callable(f) else list(f)

def walk(obj, depth=0):
    name = getattr(obj, "name", None)
    print("  " * depth + f"{obj.__class__.__name__}: {name}")
    for child in get_children(obj):
        walk(child, depth + 1)

k = kml.KML.from_string(kml_content.encode("utf-8"))

walk(k)


In [ ]:
kmz_path = Path("data/A_reg_WMS_nobor_m.kmz")
if not kmz_path.exists() or kmz_path.suffix.lower() != ".kmz":
    raise FileNotFoundError(f"Invalid KMZ file: {kmz_path}")

# Step 1: Extract KML from KMZ
with zipfile.ZipFile(kmz_path, 'r') as kmz:
    kml_files = [f for f in kmz.namelist() if f.endswith('.kml')]
    print(kml_files)
    if not kml_files:
        raise ValueError("No KML file found inside KMZ.")
        
    kml_content = kmz.read(kml_files[0]).decode('utf-8')
    print(kml.KML.parse(kml_files[0]))

# Step 2: Parse KML
k = KML.from_string(kml_content)

In [ ]:
# Create a KML object
k = kml.KML()

# Parse the KML string
k.from_string(kml_content)

In [ ]:

k = kml.KML.from_string(kml_content.encode("utf-8"))

print("KML object type:", type(k))
print("Has .features?", hasattr(k, "features"))
print("features attribute type:", type(getattr(k, "features", None)))
print("dir sample:", [a for a in dir(k) if a in ("features", "name", "ns", "namespaces", "name_spaces")])

In [ ]:
k

In [ ]:

kmz_path = Path("data/A_reg_WMS_nobor_m.kmz")

with zipfile.ZipFile(kmz_path, "r") as kmz:
    kml_files = [f for f in kmz.namelist() if f.lower().endswith(".kml")]
    if not kml_files:
        raise ValueError("No KML file found inside KMZ.")
    kml_bytes = kmz.read(kml_files[0])   # <-- keep as bytes

k = kml.KML.from_string(kml_bytes)

print("Top-level features:", len(k.features))
print("Top-level types:", [type(x).__name__ for x in k.features[:5]])


In [ ]:
import zipfile
from pathlib import Path
from fastkml import kml

kmz_path = Path("data/A_reg_WMS_nobor_m.kmz")

with zipfile.ZipFile(kmz_path, "r") as z:
    kml_name = [n for n in z.namelist() if n.lower().endswith(".kml")][0]
    kml_bytes = z.read(kml_name)

# Normalize Google Earth namespace -> OGC KML 2.2 namespace
kml_bytes = kml_bytes.replace(
    b'xmlns="http://earth.google.com/kml/2.2"',
    b'xmlns="http://www.opengis.net/kml/2.2"'
)

k = kml.KML.from_string(kml_bytes)

print("Top-level features:", len(k.features))
print("Top-level types:", [type(x).__name__ for x in k.features[:5]])

In [ ]:
!pip install lxml

In [ ]:
import zipfile
from pathlib import Path
from fastkml import kml

kmz_path = Path("data/A_reg_WMS_nobor_m.kmz")

with zipfile.ZipFile(kmz_path, "r") as z:
    kml_files = [n for n in z.namelist() if n.lower().endswith(".kml")]
    if not kml_files:
        raise ValueError("No KML file found inside KMZ.")
    kml_bytes = z.read(kml_files[0])

# Normalize older Google Earth namespace to OGC KML 2.2
kml_bytes = kml_bytes.replace(
    b'xmlns="http://earth.google.com/kml/2.2"',
    b'xmlns="http://www.opengis.net/kml/2.2"'
)

k = kml.KML.from_string(kml_bytes)

print("Top-level features:", len(k.features))
print("Top-level types:", [type(x).__name__ for x in k.features[:5]])

In [ ]:
from lxml import etree
from fastkml import kml

old_ns = "http://earth.google.com/kml/2.2"
new_ns = "http://www.opengis.net/kml/2.2"

root = etree.fromstring(kml_bytes)

def rewrite_ns(elem):
    # Rewrite element namespace in-place
    if isinstance(elem.tag, str) and elem.tag.startswith("{"):
        ns, local = elem.tag[1:].split("}")
        if ns == old_ns:
            elem.tag = f"{{{new_ns}}}{local}"
    for child in elem:
        rewrite_ns(child)

rewrite_ns(root)

# Serialize the modified XML back to bytes
fixed_bytes = etree.tostring(root, xml_declaration=True, encoding="UTF-8")

k = kml.KML.from_string(fixed_bytes)

print("Top-level features:", len(k.features))
print("Top-level types:", [type(x).__name__ for x in k.features[:5]])

In [ ]:

def walk(obj, depth=0):
    indent = "  " * depth
    nm = getattr(obj, "name", None)  # may not exist on some objects
    print(f"{indent}{type(obj).__name__} name={nm!r}")
    for child in getattr(obj, "features", []) or []:
        walk(child, depth + 1)

doc = k.features[0]  # Document
walk(doc)


In [ ]:
for child in root:
    for child in child:
        print(child)

In [ ]:

from fastkml.kml import Placemark

def iter_all(obj):
    for child in getattr(obj, "features", []) or []:
        yield child
        yield from iter_all(child)

doc = k.features[0]
placemarks = [x for x in iter_all(doc) if isinstance(x, Placemark)]

print("Placemark count:", len(placemarks))
print("First 10 names:", [getattr(pm, "name", None) for pm in placemarks[:10]])


In [ ]:
import zipfile
from pathlib import Path
from lxml import etree
import pandas as pd
import re
from datetime import datetime

def load_kml_bytes_from_kmz(kmz_path: str | Path) -> bytes:
    kmz_path = Path(kmz_path)
    with zipfile.ZipFile(kmz_path, "r") as z:
        kml_files = [n for n in z.namelist() if n.lower().endswith(".kml")]
        if not kml_files:
            raise ValueError("No KML found in KMZ.")
        return z.read(kml_files[0])

def normalize_kml_namespace(kml_bytes: bytes) -> bytes:
    # Convert Google Earth KML namespace to OGC KML 2.2 so XPath is consistent too
    return kml_bytes.replace(
        b'xmlns="http://earth.google.com/kml/2.2"',
        b'xmlns="http://www.opengis.net/kml/2.2"'
    )

def parse_lon_lat_from_coordinates(coord_text: str):
    # KML coordinate order is lon,lat[,alt]
    if not coord_text:
        return None, None
    coord_text = coord_text.strip()
    # Sometimes coordinates can be multiple tuples; for Point it’s usually one
    first = coord_text.split()[0]
    parts = first.split(",")
    if len(parts) < 2:
        return None, None
    lon = float(parts[0])
    lat = float(parts[1])
    return lon, lat

def extended_data_dict(pm, ns):
    """
    Extract ExtendedData from:
      - <ExtendedData><Data name="X"><value>Y</value></Data> ...
      - <ExtendedData><SchemaData><SimpleData name="X">Y</SimpleData> ...
    """
    out = {}

    # <Data name="..."><value>...</value></Data>
    for d in pm.xpath(".//kml:ExtendedData/kml:Data", namespaces=ns):
        key = d.get("name")
        val = d.findtext("kml:value", namespaces=ns)
        if key:
            out[key] = (val.strip() if isinstance(val, str) else val)

    # <SchemaData><SimpleData name="...">...</SimpleData>
    for sd in pm.xpath(".//kml:ExtendedData//kml:SchemaData//kml:SimpleData", namespaces=ns):
        key = sd.get("name")
        val = (sd.text or "").strip()
        if key:
            out[key] = val

    return out

def try_parse_date(value: str):
    """
    Heuristic date parse. Returns pandas.Timestamp or None.
    Adjust formats here if your data is consistent (e.g. dd/mm/yyyy).
    """
    if not value or not isinstance(value, str):
        return None
    v = value.strip()

    # common quick cleanups
    v = re.sub(r"\s+", " ", v)

    # Try pandas to_datetime which handles many formats
    try:
        dt = pd.to_datetime(v, errors="coerce", dayfirst=True, utc=False)
        if pd.isna(dt):
            return None
        return dt
    except Exception:
        return None

def find_sample_date_in_dict(d: dict):
    """
    Look for a likely sample date field in ExtendedData dict.
    """
    if not d:
        return None, None
    # prioritize keys commonly used
    candidate_keys = [k for k in d.keys() if re.search(r"(sample.*date|date.*sample|sampling.*date|\bdate\b)", k, re.I)]
    # fallback: any key containing date
    if not candidate_keys:
        candidate_keys = [k for k in d.keys() if "date" in k.lower()]

    for k in candidate_keys:
        dt = try_parse_date(d.get(k))
        if dt is not None:
            return k, dt
    return None, None

def extract_tables_from_description(desc: str):
    """
    Many KMLs embed data as HTML in <description>.
    Returns list of DataFrames (possibly empty).
    """
    if not desc or not isinstance(desc, str):
        return []
    # Some descriptions contain CDATA with HTML fragments
    try:
        tables = pd.read_html(desc)  # requires lxml installed (it is, since we use lxml)
        return tables
    except Exception:
        return []

def normalize_column_names(cols):
    return [re.sub(r"\s+", " ", str(c)).strip() for c in cols]

def pick_date_column(df: pd.DataFrame):
    """
    Identify likely date column in a table.
    """
    cols = [c.lower() for c in df.columns]
    for i, c in enumerate(cols):
        if re.search(r"(sample.*date|sampling.*date|\bdate\b)", c):
            return df.columns[i]
    return None

def kml_to_samples_df(kmz_path: str | Path) -> pd.DataFrame:
    kml_bytes = load_kml_bytes_from_kmz(kmz_path)
    kml_bytes = normalize_kml_namespace(kml_bytes)

    root = etree.fromstring(kml_bytes)

    ns = {"kml": "http://www.opengis.net/kml/2.2"}

    placemarks = root.xpath(".//kml:Placemark", namespaces=ns)
    rows = []

    for pm in placemarks:
        pm_name = pm.findtext("kml:name", namespaces=ns)

        # Coordinates (Point)
        coord_text = pm.findtext(".//kml:Point/kml:coordinates", namespaces=ns)
        lon, lat = parse_lon_lat_from_coordinates(coord_text)

        # ExtendedData fields
        ed = extended_data_dict(pm, ns)

        # Description (often HTML with tables)
        desc = pm.findtext("kml:description", namespaces=ns) or ""
        tables = extract_tables_from_description(desc)

        # If tables exist, treat each row as a sample row (best for multiple dates per location)
        if tables:
            for t in tables:
                t = t.copy()
                t.columns = normalize_column_names(t.columns)

                date_col = pick_date_column(t)
                if date_col:
                    t["sample_date"] = pd.to_datetime(t[date_col].astype(str), errors="coerce", dayfirst=True)
                else:
                    # fallback: if ExtendedData has a date, use it for all rows
                    _, dt = find_sample_date_in_dict(ed)
                    t["sample_date"] = dt

                # Add location fields
                t["placemark_name"] = pm_name
                t["lon"] = lon
                t["lat"] = lat

                # Attach ExtendedData as additional columns (won’t overwrite existing columns)
                for k, v in ed.items():
                    if k not in t.columns:
                        t[k] = v

                # Push rows
                rows.append(t)

        else:
            # No tables: single-row record from ExtendedData
            date_key, dt = find_sample_date_in_dict(ed)

            rec = {
                "placemark_name": pm_name,
                "lon": lon,
                "lat": lat,
                "sample_date": dt
            }
            # Include all ExtendedData fields
            rec.update(ed)
            rows.append(pd.DataFrame([rec]))

    if not rows:
        return pd.DataFrame()

    df = pd.concat(rows, ignore_index=True)

    # Optional: tidy sample_date
    if "sample_date" in df.columns:
        df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")

    return df

# Usage:
# df = kml_to_samples_df("data/A_reg_WMS_nobor_m.kmz")
# df.to_csv("water_samples.csv", index=False)
# df.head()

In [ ]:
# After parsing root as above and selecting placemarks...
pm = placemarks[0]

print("Name:", pm.findtext("kml:name", namespaces=ns))
print("Has ExtendedData Data nodes:", len(pm.xpath(".//kml:ExtendedData/kml:Data", namespaces=ns)))
print("Has SchemaData nodes:", len(pm.xpath(".//kml:ExtendedData//kml:SimpleData", namespaces=ns)))

desc = pm.findtext("kml:description", namespaces=ns) or ""
print("Description length:", len(desc))
print(desc[:800])  # preview

In [ ]:
import inspect

def public_attrs(obj):
    out = {}
    for name in dir(obj):
        if name.startswith("_"):
            continue
        try:
            value = getattr(obj, name)
        except Exception:
            continue
        # skip methods/functions
        if inspect.ismethod(value) or inspect.isfunction(value):
            continue
        out[name] = value
    return out

In [ ]:
def walk(obj, depth=0, keys=None):
    indent = "  " * depth
    attrs = public_attrs(obj)

    # Show a selected subset if provided, else show a compact default
    if keys is None:
        keys = ["id", "name", "description", "styleUrl", "visibility", "open", "atom_author", "atom_link", "geometry"]

    shown = {k: attrs.get(k) for k in keys if k in attrs}
    print(f"{indent}{type(obj).__name__} {shown}")

    for child in getattr(obj, "features", []) or []:
        walk(child, depth + 1, keys=keys)

doc = k.features[0]
walk(doc)

In [ ]:
def walk(obj, depth=0):
    indent = "  " * depth
    print(f"{indent}{type(obj).__name__}")

    # show the most useful fields first
    key_fields = ["id", "name", "description", "visibility", "styleUrl", "geometry", "atom_author", "atom_link"]
    # print just those if present
    dump_attrs(obj, include=key_fields)

    # recurse
    for child in getattr(obj, "features", []) or []:
        walk(child, depth + 1)

doc = k.features[0]
walk(doc)

In [ ]:
import inspect

data = []

def safe_repr(v, maxlen=999):
    """Short, safe repr to avoid dumping huge HTML/geometry objects."""
    try:
        s = repr(v)
    except Exception:
        return "<unreprable>"
    return s if len(s) <= maxlen else s[:maxlen] + "…"

def dump_attrs(obj, *, depth, include=None, exclude=None, max_items=80):
    """
    Print non-callable public attributes for a fastkml object.

    include: list/set of attribute names to show (if None, show many)
    exclude: list/set of names to hide
    max_items: cap number printed when include is None
    """
    exclude = set(exclude or [])
    include = set(include) if include is not None else None

    attrs = []
    attrs.append(("depth", depth))
    for name in dir(obj):
        if name.startswith("_"):
            continue
        if include is not None and name not in include:
            continue
        if name in exclude:
            continue

        try:
            val = getattr(obj, name)
        except Exception:
            continue

        if inspect.isroutine(val):   # skip methods/functions
            continue

        attrs.append((name, val))

    attrs.sort(key=lambda x: x[0])

    if include is None:
        attrs = attrs[:max_items]

    data.append(attrs)
    for name, val in attrs:
        print(f"    - {name}: {safe_repr(val)}")

def walk(obj, depth=0, keys=None):
    """
    Walk features tree and print a curated set of attributes per node.

    keys: list of attribute names to show for every object.
          If None, uses a sensible default.
    """
    indent = "  " * depth
    print(f"{indent}{type(obj).__name__}")

    if keys is None:
        keys = ["id", "name", "visibility", "styleUrl", "geometry", "atom_author", "atom_link", "description"]

    dump_attrs(obj, depth=depth, include=keys, exclude=set())

    for child in getattr(obj, "features", []) or []:
        walk(child, depth + 1, keys=keys)

# Usage:
doc = k.features[0]  # Document
walk(doc)

In [ ]:
for data_point in data:
    print(data_point)
    break

In [ ]:
data[3]

In [ ]:
https://www.dws.gov.za/iwqs/wms/data/WMS_pri_text_short.html